# Intelligent Fault Diagnosis Tutorial

_By the Aalto University ARotor Laboratory (Aleksanteri Hämäläinen & Aku Karhinen)_


In [ ]:
## Do not edit this cell

import math
import matplotlib.pyplot as plt
import numpy as np
import torch

import xgboost as xgb

import tqdm
import sklearn

await __import__("piplite").install("openconmo", deps=False)
from openconmo.benchmark_methods import DRS
from openconmo.utils import bandpass_filter, oneside_fft

import signal_processing
from utils import (
    HiddenPrints,
    import_polito,
    plot_confusion_matrix,
    polito_to_sklearn_format,
    squared_envelope,
)

## Dataset setup

We will use bearing fault data from the [Politecnico Di Torino test rig](https://doi.org/10.5281/zenodo.13913254). Details of the test rig can be found in [this paper](https://doi.org/10.3390/machines10010054).

Each measurement is specified by `class`, `rpm`, `radial_force`, and `axial_force`. Note that these are nominal values and do not exactly match the true values.

| Variable    | Options                                                       |
| ----------- | ------------------------------------------------------------- |
| Class       | (H) Healthy, (IR) Inner Race Damage, (OR) Outer Race Damage   |
| RPM         | 523 rpm, 937 rpm                                              |
| Radial load | 62.4 Nm, 124.8 Nm                                             |
| Axial load  | 0 Nm, 49 Nm (Not for measurements with radial load = 62.4 Nm) |


In [ ]:
## Do not edit this cell

# Import dataset
# Bearing with index 3 is the bearing faults are located on
# Measurements are all cut to 30 seconds to get a consistent dataset, because some are 60 seconds and some 30 seconds
polito_dataset, fs, frs = import_polito(bearing_index=3, measurement_max_len=30)

In [ ]:
## Do not edit this cell

# Multiply the rotating frequencies by these factors to get the characteristic fault frequencies
BPFI_factor = 10.824
BPFO_factor = 8.176

print("Characteristic fault frequencies at the two included speeds:")
print(
    "(Approximate, because the rotational speeds change slightly between measurements)"
)
print()
for k, v in frs.items():
    print(f"{k} rpm")
    print(f" BPFI: ~{v * BPFI_factor:.1f} Hz")
    print(f" BPFO: ~{v * BPFO_factor:.1f} Hz")

### Visualize measurements

Visualizing your data is an important step to verify that you are actually working with the data you think you are. It is also a good way to notice possible anomalities in the data, such as interference, broken sensors, mislabeled files, encoders running the wrong way, weird amplitude differences, etc.

Below, we plot 0.5 seconds from each measurement. Feel free to uncomment the lines with different signal analysis methods from OpenConMo to view the data in different ways.


In [ ]:
# Sort the dataset by class
signals = sorted(polito_dataset.items(), key=lambda kv: kv[0])

# Plotting stuff
n = len(signals)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(
    nrows, ncols, figsize=(18, 4 * nrows), sharex=False, sharey=True
)
axes = np.atleast_1d(axes).ravel()

# Loop through all measurements
for ax, (key, signal) in zip(axes, signals):
    class_label, rpm, radial_load, axial_load = key

    # Preprocess signal #

    ## HERE:  Optionally, plot the random (or discrete) part of the DRS
    # signal, signal_discrete = DRS(signal, 3000, 12_000)

    # HERE: Optionally, bandpass filter the signal
    # signal = openconmo.utils.bandpass_filter(signal, fs, 3000, 1200)

    ## HERE: Optionally, plot the squared envelope
    # signal = squared_envelope(signal)

    # Cut signal to make it easier to see the details
    signal = signal[2 * fs : 2 * fs + (fs // 4)]
    x = np.arange(len(signal)) / fs

    ## HERE: Optionally, plot the spectrum
    # x = np.fft.rfftfreq(len(signal), 1/fs)[1:]
    # signal = np.abs(np.fft.rfft(signal))[1:]

    # Plot #

    ax.plot(x, signal, linewidth=0.7)
    ax.set_title(
        f"{class_label} | {rpm} rpm | {radial_load} kN | {axial_load} kN", fontsize=10
    )
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("m/s²")

plt.tight_layout()
plt.show()

Below, you can view a single signal in more detail.


In [ ]:
# HERE: You can change which signal to plot
fault_vis = "IR"  # H/OR/IR
rpm_vis = 937  # 523/937
radial_load_vis = 124.8  # 62.4/124.8
axial_load_vis = 0.0  # 0.0/49.0

# Get measurement
s1 = polito_dataset[(fault_vis, rpm_vis, radial_load_vis, axial_load_vis)]

# Preprocess signal
# HERE: Add a bandpass filter if you want
# s1 = bandpass_filter(s1, fs, 3000, 1500)
s1 = squared_envelope(s1[:fs])  # Keep only the first second
f, X = oneside_fft(s1, fs)

# Plot
plt.figure(figsize=(14, 4))
plt.suptitle(
    f"FFT of the squared envelope of the signal: {class_label} | {rpm} rpm | {radial_load} kN | {axial_load} kN"
)
plt.xlabel("Frequency (Hz)")
plt.plot(f[1 : len(f) // 8], X[1 : len(f) // 8])

for i in range(1, 5):
    plt.axvline(frs[rpm_vis] * BPFI_factor * i, color="r", linestyle="--", alpha=0.7)
    plt.axvline(frs[rpm_vis] * BPFO_factor * i, color="g", linestyle="--", alpha=0.7)

### Signal Transformations

Depending on the machine, sensors, and ML/DL methods, different types of preprocessing will be appropriate. The cell below lets you choose between DRS, bandpass filtering, and/or using the squared envelope.

TODO: Start with nothing


In [ ]:
# HERE: You can choose which preprocessing steps to apply by changing the methods argument
polito_dataset_preprocessed = signal_processing.preprocess_signals(
    polito_dataset,
    methods=[
        # "DRS",
        # "bandpass_filter",
        "squared_envelope",
    ],
)

### Convert dataset to Scikit-Learn compatible format

[Scikit-learn](scikit-learn.org) is a popular and convenient Python library for machine learning. We will use some classifiers from it here, so let us create training and testing datasets in the appropriate shapes.


In [ ]:
# HERE: You can change which measurements to use for training and testing by changing the rpms, radial_loads, and axial_loads arguments
#   RPM options: 523, 937
#   Radial load options (Nm): 62.4, 124.8
#   Axial load options (Nm): 0, 49 (49 Nm only for 124.8 Nm load)

X_train_raw, y_train, X_train_rpm = polito_to_sklearn_format(
    polito_dataset_preprocessed,
    rpms=[523],
    # rpms=[937],
    radial_loads=[62.4],
    axial_loads=[0],
)
X_test_raw, y_test, X_test_rpm = polito_to_sklearn_format(
    polito_dataset_preprocessed,
    rpms=[523],
    # rpms=[937],
    radial_loads=[124.8],
    axial_loads=[0],
)
# X_test_raw = X_test_raw * 2

### Feature extraction

Most ML methods work better with extracted features instead of raw signals. In the cell below, you can select from a set of defined methods. You can also define one or more of your own methods.

First however, implement the widely used root mean square yourself. The RMS of a discrete signal \(x\) with \(N\) samples is defined as follows:

$$\mathrm{RMS}(x)=\sqrt{\frac{1}{N}\sum_{n=1}^{N}x_n^2}$$


In [ ]:
# CODE HERE
def rms(samples, rpms):
    return ...


# CODE HERE: You can define your own feature extractor
# def my_own_feature(samples):
#     return ...  # implement your own feature here


def get_features(samples, rpms, features):
    all_feature_extractors = {
        "mean": signal_processing.mean,
        "rms": rms,
        "variance": signal_processing.variance,
        "skewness": signal_processing.skewness,
        "kurtosis": signal_processing.kurtosis_,
        "peak_to_peak": signal_processing.peak_to_peak,
        "crest_factor": signal_processing.crest_factor,
        "BPFO_1": signal_processing.BPFO_1,
        "BPFO_2": signal_processing.BPFO_2,
        "BPFO_3": signal_processing.BPFO_3,
        "BPFI_1": signal_processing.BPFI_1,
        "BPFI_2": signal_processing.BPFI_2,
        "BPFI_3": signal_processing.BPFI_3,
        # "my_own_feature": my_own_feature,
    }

    # Use specified feature extractors
    new_samples = [all_feature_extractors[key](samples, rpms) for key in features]

    return np.stack(new_samples, axis=1)


# HERE: You can choose which features to use by changing the features_to_use list
features_to_use = [
    "mean",
    "rms",
    "variance",
    "skewness",
    "kurtosis",
    "peak_to_peak",
    "crest_factor",
    # "my_own_feature",
    # "BPFO_1",
    # "BPFO_2",
    # "BPFO_3",
    # "BPFI_1",
    # "BPFI_2",
    # "BPFI_3",
]

X_train = get_features(X_train_raw, X_train_rpm, features_to_use)
X_test = get_features(X_test_raw, X_test_rpm, features_to_use)

## Classification

### XGBoost

The first method we will use is [XGBoost](https://xgboost.readthedocs.io). XGBoost has its own library and not from Scikit-learn, but it does provide a [Scikit-Learn estimator interface](https://xgboost.readthedocs.io/en/release_3.2.0/python/sklearn_estimator.html), meaning it can be used very similarly to any other classifiers from Scikit-learn.

XGBoost describes itself as:

> XGBoost is an optimized distributed gradient boosting library designed to be highly efficient, flexible and portable. It implements machine learning algorithms under the Gradient Boosting framework. XGBoost provides a parallel tree boosting (also known as GBDT, GBM) that solve many data science problems in a fast and accurate way. The same code runs on major distributed environment (Hadoop, SGE, MPI) and can solve problems beyond billions of examples.

It is a good baseline for nearly any classification problem when the data is structured or tabular (i.e. not raw time-series or image data).


In [ ]:
clf = xgb.XGBClassifier(early_stopping_rounds=6, max_depth=3, learning_rate=0.1)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

print()
print(f"Test set accuracy: {clf.score(X_test, y_test)*100:.2f}%")

plot_confusion_matrix(clf, X_test, y_test)

### Support Vector Machine (SVM)

Support vector machines are a classic method for classification. We will use the [Support Vector Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) (since we have more than 2 classes) from Scikit-learn.


In [ ]:
from sklearn.svm import SVC

clf = SVC(kernel="linear", C=1)
clf.fit(X_train, y_train)

acc = clf.score(X_test, y_test)
print(f"Test set accuracy: {acc*100:.2f}%")

plot_confusion_matrix(clf, X_test, y_test)

### Multilayer Perceptron (MLP)

[MLPs](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) are the first step towards deep learning. MLPs can be constructed with other more deep learning oriented libraries, such as PyTorch or TensorFlow, but Scikit-learn also provides an easy way to make them.

In the cell below, use the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) to make an MLP with two hidden layers sized 32 and 8 (or whatever you feel like) and early stopping after 10 batches of no improvement.

_Run the cell multiple times to see how the results change_


In [ ]:
from sklearn.neural_network import MLPClassifier

# CODE HERE: Define an MLPClassifier with early stopping and a reasonable architecture for this problem
# clf = ...
clf.fit(X_train, y_train)

acc = clf.score(X_test, y_test)
print(f"Test set accuracy: {acc*100:.2f}%")

plot_confusion_matrix(clf, X_test, y_test)

Deep learning methods, or more specifically methods using stochastic gradient decent, can get very different results from one training run to another. Because of this, it is important to report the average of multiple runs and the standard deviation to show that the results weren't due to good luck.


In [ ]:
mlp_results = []
# CODE HERE: How many repetitions do you think are enough?
for i in range(...):
    clf = ...  # CODE HERE: Copy paste your MLPClassifier definition
    clf.fit(X_train, y_train)

    acc = clf.score(X_test, y_test)
    mlp_results.append(acc)

print(
    f"Test set accuracy: {np.mean(mlp_results)*100:.2f}% ± {np.std(mlp_results)*100:.2f}%"
)
print(f"Minimum accuracy: {np.min(mlp_results)*100:.2f}%")
print(f"Maximum accuracy: {np.max(mlp_results)*100:.2f}%")

## Deep learning version

One advantage deep learning methods have is the ability to make use of raw data (e.g. time-series or images). This reduces the need for manual feature engineering, which is beneficial because it is not always clear what are the best features to use.

In the cells below, you have the possibility to run two different models. The first is [ZoomCNN](https://doi.org/10.1016/j.ymssp.2023.110865), which is built to imitate how traditional signal analysis methods work. The next is [WDCNN](https://doi.org/10.3390/s17020425), an older but still quite widely used benchmark CNN method for condition monitoring.

For a tutorial on how to make training loops for DL models, see [here](https://wandb.ai/wandb_fc/tips/reports/How-To-Write-Efficient-Training-Loops-in-PyTorch--VmlldzoyMjg4OTk5?galleryTag=pytorch).


In [ ]:
## Do not edit this cell

# Create PyTorch dataloaders from the previous data splits

dataloader_train = torch.utils.data.DataLoader(
    list(zip(torch.Tensor(X_train_raw), torch.Tensor(y_train))),
    shuffle=True,
    batch_size=32,
)
dataloader_test = torch.utils.data.DataLoader(
    list(zip(torch.Tensor(X_test_raw), torch.Tensor(y_test))), batch_size=64
)

In [ ]:
from torch_models import WDCNN, ZoomCNN

# HERE: You can choose which model to use
# model = WDCNN()
model = ZoomCNN()


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Epochs
for epoch in range(6):

    # Training #

    for i, (X_batch, y_batch) in enumerate(dataloader_train):
        model.train()

        # Add channel dimension and convert to float
        X_batch = X_batch.unsqueeze(1).float()
        y_batch = y_batch.long()

        # Reset gradients and set model to training mode

        # Forward Pass
        y_hat = model(X_batch)

        # Compute Loss and Perform Back-propagation
        loss = loss_fn(y_hat, y_batch)
        loss.backward()

        # Update Optimizer
        optimizer.step()

        # Validation
        if i % 20 == 0:
            print(f"Epoch {epoch+1} | Batch {i+1: <2} | Loss: {loss.item():.4f}")

        # Reset
        optimizer.zero_grad()
        torch.cuda.empty_cache()

    # Validation #

    loss_validation = 0
    acc_validation = 0
    for i, (X_batch, y_batch) in enumerate(dataloader_test):
        model.eval()

        # Add channel dimension and convert to float
        X_batch = X_batch.unsqueeze(1).float()
        y_batch = y_batch.long()

        # Forward Pass
        with torch.no_grad():
            y_hat = model(X_batch)

        # Compute Loss and Perform Back-propagation
        loss = loss_fn(y_hat, y_batch)
        loss_validation += loss.item()
        acc_validation += (y_hat.argmax(dim=1) == y_batch).float().mean().item()

    print(
        f"   Epoch {epoch+1} | Validation Loss: {loss_validation:.4f} | Validation Accuracy: {acc_validation / len(dataloader_test) * 100:.2f}%"
    )